# Tasty Samplers with chatsnack 🍿

Is popcorn crunchy? Does caramel corn count as dessert? Sometimes we want an answer we can use in an `if` statement.

Let's try a `Sampler`. We give it something to look at and a question, and `ask()` brings back a `Sample` with our answer.

## Got snack?

Install `chatsnack[typesafe]` and add `TYPESAFE_API_KEY` to your `.env` file. Samplers use TypeSafe's Jev model. We'll also use your OpenAI key when we get to the Chat example.

## First bite

Let's start with a bowl of buttered popcorn.

In [ ]:
from chatsnack import Chat, Question, Sampler

sample = Sampler(data="buttered popcorn").ask("Is this crunchy?")
print(sample.answer.yes)


Our answer lives at `sample.answer`. `.yes` gives us a Boolean we can branch on; `.score` gives us the probability of yes, from 0 to 1.

Try swapping the popcorn for melted ice cream. Same question, very different snack.

## Save a little crunch

We could ask about crunch all day. Let's give the question a name and save it for the next snack. The `yes` description lets us say what we mean by crunchy.

Then we'll make a `SnackCheck` Sampler. `{snack}` leaves room for whatever we're eating, and `{question.crunchy}` picks up our saved Question each time we call `ask()`.

In [ ]:
crunchy = Question(
    name="crunchy",
    question="Is this crunchy?",
    yes="It makes a crisp cracking sound when bitten.",
)
crunchy.save()

review = Sampler(name="SnackCheck", data="{snack}", questions=["{question.crunchy}"])
print(review.yaml)
review.save()


### Yummy YAML

Here's our saved `SnackCheck`. Just enough to remember what we're asking:

```yaml
data: "{snack}"
questions:
  - "{question.crunchy}"
```

We can edit the saved YAML in a text editor, just like our chats. Changing the saved `crunchy` Question changes what `SnackCheck` asks next time.

Let's load the Sampler back and give it some popcorn. `.load()` reads the file; `snack=...` fills in the blank.

In [ ]:
review = Sampler(name="SnackCheck")
review.load()
sample = review.ask(snack="popcorn")
print(sample.answer.choice, sample.answer.score)


## The sampler platter

Crunch isn't everything. Let's ask what kind of food we've got and how sweet it is, all in one call.

`choices` gives us options to pick from. `levels` gives us an ordered scale. We'll name the questions so it's easy to find each answer afterward.

In [ ]:
category = Question(name="category", question="What kind of food is this?", choices=["snack", "meal"])
sweetness = Question(name="sweetness", question="How sweet is this?", levels=["Not sweet", "Very sweet"])

sample = review.ask(questions=["{question.crunchy}", category, sweetness], snack="popcorn")
print(sample.answers["category"].choice)
print(sample.answers["sweetness"].score)


`sample.answers["category"]` still finds the category if we shuffle the questions around. Handy once our platter gets bigger.

`sample.answer` always means the first answer, even when we ask several questions. Here that's crunchiness. For sweetness, `.choice` gives us the most likely level and `.score` gives us the model's weighted score.

## Snack fillings, now with crunch

Remember chat fillings? Our saved Sampler can be a filling too. Let's give a menu-writing Chat the crunch verdict.

Both fillings below use the same `SnackCheck` result. Asking for its choice and score runs the Sampler once for this Chat call.

In [ ]:
description = Chat(
    "Write a tempting one-sentence menu description for {snack}. "
    "Crunchy: {sampler.SnackCheck.crunchy.choice}. "
    "Probability of yes: {sampler.SnackCheck.crunchy.score}."
)
print(description.ask(snack="popcorn"))


## Save some for later

Want to try that same bowl again later? `from_sample()` makes a Sampler from the popcorn data and the three questions we just used. It also keeps the model version that answered.

We can keep tinkering with our saved `crunchy` Question without changing this copy. Running it again may give different answers, but we'll be asking about the same popcorn.

In [ ]:
replay = Sampler.from_sample(sample, name="PopcornReview")
replay.save()
repeated = replay.ask()
print(repeated.answers["category"].choice)
print(repeated.model, repeated.usage)
